In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_addons as tfa
# import matplotlib.pyplot as plt
# import numpy as np

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Device:', tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
except:
    strategy = tf.distribute.get_strategy()
print('Number of replicas:', strategy.num_replicas_in_sync)

AUTOTUNE = tf.data.experimental.AUTOTUNE
    
print(tf.__version__)

Number of replicas: 1
2.10.1


c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you wa

In [3]:
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [4]:
COMICS_FILENAMES = tf.io.gfile.glob(str('./real2anime/trainA/*.jpg'))
print('Comics Files:', len(COMICS_FILENAMES))

FACES_FILENAMES = tf.io.gfile.glob(str('./real2anime/trainB/*.jpg'))
print('Faces  Files:', len(FACES_FILENAMES))

Comics Files: 3400
Faces  Files: 3400


In [5]:
REAL_FACES_FILENAMES = tf.io.gfile.glob(str('./real2anime/testA/*.jpg'))
print('Faces  Files:', len(REAL_FACES_FILENAMES))

Faces  Files: 100


In [6]:
import random
EXTRA_COMIC = random.sample(COMICS_FILENAMES,len(REAL_FACES_FILENAMES))
# Random Shuffle IF needed
import random
random.shuffle(FACES_FILENAMES)
random.shuffle(COMICS_FILENAMES)

In [7]:
ALL_FACES = FACES_FILENAMES + REAL_FACES_FILENAMES
ALL_COMICS = EXTRA_COMIC + COMICS_FILENAMES
print(len(ALL_FACES), len(ALL_COMICS))

3500 3500


In [8]:
# import shutil
# import os
# import pathlib
# import zipfile
# from glob import glob
# import logging
import warnings
# import datetime

# import numpy as np 
# import pandas as pd

import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import matplotlib.style as style
# from PIL import Image
# import seaborn as sns

# from tqdm import tqdm
# from itertools import chain
# from collections import Counter, defaultdict


# from keras.models import Sequential
# from keras.layers import Dense, Dropout, Flatten, BatchNormalization
# from keras.layers import Conv2D, MaxPooling2D


warnings.filterwarnings('ignore')



In [9]:
#Configuration

BATCH_SIZE = 4

BUFFER_SIZE = 100
IMG_HEIGHT, IMG_WIDTH = 128, 128
CROP_SIZE = 128
TEST_SIZE = 0.01
CHANNELS = 3
HEIGHT = 128
WIDTH = 128
LABEL_SMOOTHING = 0.1

TRANSFORMER_BLOCKS = 6
GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 4e-4

In [10]:
from sklearn.model_selection import train_test_split
train_comics_paths, test_comics_paths, train_faces_paths, test_faces_paths = train_test_split(ALL_COMICS, ALL_FACES, test_size=TEST_SIZE, random_state=420)

In [11]:
def load(comic_path,face_path):
    comic = tf.io.read_file(comic_path)
    comic = tf.image.decode_jpeg(comic, channels=3)
    


    face = tf.io.read_file(face_path)
    face = tf.image.decode_jpeg(face, channels=3)
    

    return comic ,face

def normalize(comic, face):
    """ Normalize the images to [-1, 1]
    """
    # YOUR CODE HERE
    comic = (tf.cast(comic, tf.float32) /255.0 *2) -1
    face = (tf.cast(face, tf.float32) /255.0 *2) -1
    return comic, face

def random_crop(comic, face, height, width):
    """ Stack the two images on top of each other
        Then crop together.
    """
    stacked_image = tf.stack([comic, face], axis=0)
    cropped_image = tf.image.random_crop(stacked_image, 
                                         size=[2, height, width, 3])

    return cropped_image[0], cropped_image[1]

def resize(comic, face, height, width):
    """ Resize the two image to heigh, width 
    """
    comic = tf.image.resize(comic, 
                           [height, width],
                           method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    face = tf.image.resize(face, 
                               [height, width],
                               method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

    return comic, face
    
@tf.function()
def random_jitter(comic, face):
    comic, face = resize(comic, face, IMG_HEIGHT, IMG_WIDTH)
    comic, face = random_crop(comic, face, CROP_SIZE, CROP_SIZE)

    # Augmentation to random flip
    if tf.random.uniform(()) > 0.5: 
        comic = tf.image.flip_left_right(comic)
        face = tf.image.flip_left_right(face)

    return comic, face

def preprocess(comic_path, face_path):
    comic, face = load(comic_path, face_path)



    return comic, face
def imgaug(comic,face):
    comic,face = random_jitter(comic,face)
    comic, face = normalize(comic, face)
    
    
    return comic, face


In [12]:

AUTOTUNE = tf.data.experimental.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((train_comics_paths,train_faces_paths))
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE).cache()
train_ds = train_ds.repeat().shuffle(buffer_size=1000,reshuffle_each_iteration=True).map(imgaug, num_parallel_calls=tf.data.experimental.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_comics_paths, test_faces_paths))
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE).cache()
test_ds = test_ds.repeat().shuffle(buffer_size=50).map(imgaug, num_parallel_calls=tf.data.experimental.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [13]:
# Model Building

conv_initializer = tf.random_normal_initializer(mean=0.0, stddev=0.02)
gamma_initializer = tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02)
    
def encoder_block(input_layer, filters, size=3, strides=2, apply_instancenorm=True,apply_spectral=False, activation=layers.LeakyReLU(0.2), name='block_x'):
    if (apply_spectral) & (apply_instancenorm ==False):
        block = tfa.layers.SpectralNormalization(layers.Conv2D(filters, size, 
                     strides=strides, 
                     padding='same', 
                     use_bias=False, 
                     kernel_initializer=conv_initializer, 
                     name=f'encoder_{name}'))(input_layer)
    else: 
        block =  layers.Conv2D(filters, size, 
                     strides=strides, 
                     padding='same', 
                     use_bias=False, 
                     kernel_initializer=conv_initializer, 
                     name=f'encoder_{name}')(input_layer)

    if apply_instancenorm:
        block = tfa.layers.InstanceNormalization(gamma_initializer=gamma_initializer)(block)
        # block = gamma_initializer=gamma_initializer)(block)
        
        
    block = activation(block)

    return block

def transformer_block(input_layer, size=3, strides=1, name='block_x'):
    filters = input_layer.shape[-1]
    
    block = layers.Conv2D(filters, size, strides=strides, padding='same', use_bias=False, 
                     kernel_initializer=conv_initializer, name=f'transformer_{name}_1')(input_layer)
    # block = tfa.layers.InstanceNormalization(gamma_initializer=gamma_initializer)(block)
    block = layers.LeakyReLU(0.2)(block)
    
    block = layers.Conv2D(filters, size, strides=strides, padding='same', use_bias=False, 
                     kernel_initializer=conv_initializer, name=f'transformer_{name}_2')(block)
    # block = tfa.layers.InstanceNormalization(gamma_initializer=gamma_initializer)(block)
    
    block = layers.Add()([block, input_layer])

    return block

def decoder_block(input_layer, filters, size=3, strides=2, apply_instancenorm=True,apply_spectral=False, name='block_x'):
    
    if (apply_spectral) & (apply_instancenorm ==False):
        block = tfa.layers.SpectralNormalization(layers.Conv2DTranspose(filters, size, 
                              strides=strides, 
                              padding='same', 
                              use_bias=False, 
                              kernel_initializer=conv_initializer, 
                              name=f'decoder_{name}'))(input_layer)
    else:
    
        block = layers.Conv2DTranspose(filters, size, 
                              strides=strides, 
                              padding='same', 
                              use_bias=False, 
                              kernel_initializer=conv_initializer, 
                              name=f'decoder_{name}')(input_layer)

    if apply_instancenorm:
        block = tfa.layers.InstanceNormalization(gamma_initializer=gamma_initializer)(block)

    block = layers.LeakyReLU()(block)
    
    return block


###PatchGAN Discriminator

def Discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)
    gamma_init = keras.initializers.RandomNormal(mean=0.0, stddev=0.02)

    inp = layers.Input(shape=[128, 128, 3], name='input_image')

    x = inp
    enc_1 = encoder_block(x, 64,  4, 2, apply_instancenorm=False, activation=layers.LeakyReLU(0.2), name='block_1')
    enc_2 = encoder_block(enc_1, 128, 4, 2, apply_instancenorm=True,apply_spectral=False, activation=layers.LeakyReLU(0.2), name='block_2') 
    enc_3 = encoder_block(enc_2, 256, 4, 2, apply_instancenorm=True,apply_spectral=False, activation=layers.LeakyReLU(0.2), name='block_3') 

    zero_pad1 = layers.ZeroPadding2D()(enc_3) # (bs, 34, 34, 256)
    conv = layers.Conv2D(512, 4, strides=1,
                         kernel_initializer=initializer,
                         use_bias=False)(zero_pad1) # (bs, 31, 31, 512)

    norm1 = tfa.layers.InstanceNormalization(gamma_initializer=gamma_init)(conv)

    leaky_relu = layers.LeakyReLU(0.2)(norm1)

    zero_pad2 = layers.ZeroPadding2D()(leaky_relu) # (bs, 33, 33, 512)

    last = layers.Conv2D(1, 4, strides=1,
                         kernel_initializer=initializer)(zero_pad2) # (bs, 30, 30, 1)

    return tf.keras.Model(inputs=inp, outputs=last)
tf.keras.utils.plot_model(Discriminator(), show_shapes=True, dpi=64)

def generator_fn(height=HEIGHT, width=WIDTH, channels=CHANNELS, transformer_blocks=TRANSFORMER_BLOCKS):
    OUTPUT_CHANNELS = 3
    inputs = layers.Input(shape=[height, width, channels], name='input_image')

    # Encoder
    enc_1 = encoder_block(inputs, 64,  7, 1, apply_instancenorm=False, activation=layers.LeakyReLU(0.2), name='block_1') # (bs, 256, 256, 64)
    enc_2 = encoder_block(enc_1, 128, 3, 2, apply_instancenorm=True,apply_spectral=False, activation=layers.LeakyReLU(0.2), name='block_2')   # (bs, 128, 128, 128)
    enc_3 = encoder_block(enc_2, 256, 3, 2, apply_instancenorm=True,apply_spectral=False, activation=layers.LeakyReLU(0.2), name='block_3')   # (bs, 64, 64, 256)
    
    # Transformer
    x = enc_3
    for n in range(transformer_blocks):
        x = transformer_block(x, 3, 1, name=f'block_{n+1}') # (bs, 64, 64, 256)

    # Decoder
    x_skip = layers.Concatenate(name='enc_dec_skip_1')([x, enc_3]) # encoder - decoder skip connection
    
    dec_1 = decoder_block(x_skip, 128, 3, 2, apply_instancenorm=True,apply_spectral=False, name='block_1') # (bs, 128, 128, 128)
    x_skip = layers.Concatenate(name='enc_dec_skip_2')([dec_1, enc_2]) # encoder - decoder skip connection
    
    dec_2 = decoder_block(x_skip, 64,  3, 2, apply_instancenorm=True,apply_spectral=False, name='block_2') # (bs, 256, 256, 64)
    x_skip = layers.Concatenate(name='enc_dec_skip_3')([dec_2, enc_1]) # encoder - decoder skip connection

    outputs = last = layers.Conv2D(OUTPUT_CHANNELS, 7, 
                              strides=1, padding='same', 
                              kernel_initializer=conv_initializer, 
                              use_bias=False, 
                              activation='tanh', 
                              name='decoder_output_block')(x_skip) # (bs, 256, 256, 3)

    generator = keras.Model(inputs, outputs)
    
    return generator

sample_generator = generator_fn()

tf.keras.utils.plot_model(sample_generator, show_shapes=True, dpi=64)



with strategy.scope():
    comics_generator = generator_fn(height=None, width=None, transformer_blocks=TRANSFORMER_BLOCKS) 
    faces_generator = generator_fn(height=None, width=None, transformer_blocks=TRANSFORMER_BLOCKS)

    comics_discriminator = Discriminator() # differentiates real Comics and generated Comic
    faces_discriminator = Discriminator() # differentiates real photos and generated photos

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.
You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [14]:
class CycleGan(keras.Model):
    def __init__(
        self,
        comics_generator,
        faces_generator,
        comics_discriminator,
        faces_discriminator,
        lambda_cycle=10,
    ):
        super(CycleGan, self).__init__()
        self.m_gen = comics_generator
        self.p_gen = faces_generator
        self.m_disc = comics_discriminator
        self.p_disc = faces_discriminator
        self.lambda_cycle = lambda_cycle
        
    def compile(
        self,
        m_gen_optimizer,
        p_gen_optimizer,
        m_disc_optimizer,
        p_disc_optimizer,
        gen_loss_fn,
        disc_loss_fn,
        cycle_loss_fn,
        identity_loss_fn
    ):
        super(CycleGan, self).compile()
        self.m_gen_optimizer = m_gen_optimizer
        self.p_gen_optimizer = p_gen_optimizer
        self.m_disc_optimizer = m_disc_optimizer
        self.p_disc_optimizer = p_disc_optimizer
        self.gen_loss_fn = gen_loss_fn
        self.disc_loss_fn = disc_loss_fn
        self.cycle_loss_fn = cycle_loss_fn
        self.identity_loss_fn = identity_loss_fn
        
    def train_step(self, batch_data):
        real_comic, real_face = batch_data
        
        with tf.GradientTape(persistent=True) as tape:
            # Face to Comic back to Face
            fake_comic = self.m_gen(real_face, training=True)
            cycled_face = self.p_gen(fake_comic, training=True)

            # Comic to Face back to Comic
            fake_face = self.p_gen(real_comic, training=True)
            cycled_comic = self.m_gen(fake_face, training=True)

            # generating itself
            same_comic = self.m_gen(real_comic, training=True)
            same_face = self.p_gen(real_face, training=True)

            # discriminator used to check, inputing real images
            disc_real_comic = self.m_disc(real_comic, training=True)
            disc_real_face = self.p_disc(real_face, training=True)

            # discriminator used to check, inputing fake images
            disc_fake_comic = self.m_disc(fake_comic, training=True)
            disc_fake_face = self.p_disc(fake_face, training=True)

            # evaluates generator loss
            comic_gen_loss = self.gen_loss_fn(disc_fake_comic)
            face_gen_loss = self.gen_loss_fn(disc_fake_face)

            # evaluates total cycle consistency loss
            total_cycle_loss = self.cycle_loss_fn(real_comic, cycled_comic, self.lambda_cycle) + self.cycle_loss_fn(real_face, cycled_face, self.lambda_cycle)

            # evaluates total generator loss
            total_comic_gen_loss = comic_gen_loss + total_cycle_loss + self.identity_loss_fn(real_comic, same_comic, self.lambda_cycle)
            total_face_gen_loss = face_gen_loss + total_cycle_loss + self.identity_loss_fn(real_face, same_face, self.lambda_cycle)

            # evaluates discriminator loss
            comic_disc_loss = self.disc_loss_fn(disc_real_comic, disc_fake_comic)
            face_disc_loss = self.disc_loss_fn(disc_real_face, disc_fake_face)

        # Calculate the gradients for generator and discriminator
        comic_generator_gradients = tape.gradient(total_comic_gen_loss,
                                                  self.m_gen.trainable_variables)
        face_generator_gradients = tape.gradient(total_face_gen_loss,
                                                  self.p_gen.trainable_variables)

        comic_discriminator_gradients = tape.gradient(comic_disc_loss,
                                                      self.m_disc.trainable_variables)
        face_discriminator_gradients = tape.gradient(face_disc_loss,
                                                      self.p_disc.trainable_variables)

        # Apply the gradients to the optimizer
        self.m_gen_optimizer.apply_gradients(zip(comic_generator_gradients,
                                                 self.m_gen.trainable_variables))

        self.p_gen_optimizer.apply_gradients(zip(face_generator_gradients,
                                                 self.p_gen.trainable_variables))

        self.m_disc_optimizer.apply_gradients(zip(comic_discriminator_gradients,
                                                  self.m_disc.trainable_variables))

        self.p_disc_optimizer.apply_gradients(zip(face_discriminator_gradients,
                                                  self.p_disc.trainable_variables))
        
        return {
            "comic_gen_loss": total_comic_gen_loss,
            "face_gen_loss": total_face_gen_loss,
            "comic_disc_loss": comic_disc_loss,
            "face_disc_loss": face_disc_loss
        }


## Loss Def

####Discriminator Loss with label smoothing

with strategy.scope():
    def discriminator_loss(real, generated):
        real_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE,label_smoothing=LABEL_SMOOTHING)(tf.ones_like(real), real)

        generated_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE,label_smoothing=LABEL_SMOOTHING)(tf.zeros_like(generated), generated)

        total_disc_loss = real_loss + generated_loss

        return total_disc_loss * 0.5

####Generator Loss

with strategy.scope():
    def generator_loss(generated):
        return tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)(tf.ones_like(generated), generated)

####Cycle Loss

with strategy.scope():
    def calc_cycle_loss(real_image, cycled_image, LAMBDA):
        loss1 = tf.reduce_mean(tf.abs(real_image - cycled_image))

        return LAMBDA * loss1

#### Identity Loss

with strategy.scope():
    def identity_loss(real_image, same_image, LAMBDA):
        loss = tf.reduce_mean(tf.abs(real_image - same_image))
        return LAMBDA * 0.5 * loss

#### Optimizer

with strategy.scope():
    comic_generator_optimizer = tf.keras.optimizers.Adam(GENERATOR_LR, beta_1=0.5)
    face_generator_optimizer = tf.keras.optimizers.Adam(GENERATOR_LR, beta_1=0.5)

    comic_discriminator_optimizer = tf.keras.optimizers.Adam(DISCRIMINATOR_LR, beta_1=0.5)
    face_discriminator_optimizer = tf.keras.optimizers.Adam(DISCRIMINATOR_LR, beta_1=0.5)

In [15]:
with strategy.scope():
    cycle_gan_model = CycleGan(
        comics_generator, faces_generator, comics_discriminator, faces_discriminator
    )

    cycle_gan_model.compile(
        m_gen_optimizer = comic_generator_optimizer,
        p_gen_optimizer = face_generator_optimizer,
        m_disc_optimizer = comic_discriminator_optimizer,
        p_disc_optimizer = face_discriminator_optimizer,
        gen_loss_fn = generator_loss,
        disc_loss_fn = discriminator_loss,
        cycle_loss_fn = calc_cycle_loss,
        identity_loss_fn = identity_loss
    )


## Aux Function

def generate_images(model, test_input,epoch,save_path = ''):
    prediction = model(test_input, training=True)
    plt.figure(figsize=(20,20))

    display_list = [test_input[0], prediction[0]]

    title = ['Input Image', 'Predicted Image']
    for i in range(2):
        plt.subplot(1, 2, i+1)
        plt.title(title[i])
        # getting the pixel values between [0, 1] to plot it.
        plt.imshow(display_list[i] * 0.5 + 0.5)
        plt.axis('off')
    if save_path != '':
        plt.savefig(f'{save_path}/{epoch}.jpg', bbox_inches='tight')
        

    plt.show()

In [16]:
## Custom CalLBack

from tensorflow.keras.callbacks import Callback
class GANMonitor(Callback):
    def __init__(self, num_img=1,save_every_epoch=4,watch_every_epoch=2,save_dir='./'):
        self.num_img = num_img
        self.save_every_epoch = save_every_epoch
        self.watch_every_epoch = watch_every_epoch
        self.save_dir = save_dir

    def on_epoch_end(self, epoch, logs=None):
        # Generate Image every second epoch
        if epoch % self.watch_every_epoch == 0:
            for example_input, example_target in test_ds.take(self.num_img): #Generate Comic from Photo
                generate_images(comics_generator, example_target,epoch,save_path = '')
            for example_input, example_target in test_ds.take(self.num_img): #Generate Photo from Comic
                generate_images(faces_generator, example_input,epoch,save_path='')
        if (epoch > 0) & (epoch % self.save_every_epoch == 0):
            #Save Generator
            comics_generator.save(f"{self.save_dir}/AnimeGenTrain_{epoch}.h5",save_format='h5',overwrite=True)
            faces_generator.save(f"{self.save_dir}/FaceGenTrain_{epoch}.h5",save_format='h5',overwrite=True)
    
            

In [17]:
STEP=len(train_comics_paths)//BATCH_SIZE
print(STEP)

866


In [18]:
history = cycle_gan_model.fit(
        train_ds,
        epochs=100,verbose=True,callbacks=[GANMonitor(save_every_epoch=5,watch_every_epoch=4)],steps_per_epoch=STEP,initial_epoch=0
    )

Epoch 1/100


NotFoundError: Graph execution error:

Detected at node 'model_2/decoder_output_block/Conv2D' defined at (most recent call last):
    File "c:\Users\dikib\miniconda3\envs\tf\lib\runpy.py", line 197, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\dikib\miniconda3\envs\tf\lib\runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\traitlets\config\application.py", line 1043, in launch_instance
      app.start()
    File "c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\ipykernel\kernelapp.py", line 736, in start
      self.io_loop.start()
    File "c:\Users\dikib\miniconda3\envs\tf\lib\site-packages\tornado\platform\asyncio.py", line 195, in start
      self.asyncio_loop.run_forever()
    File "c:\Users\dikib\miniconda3\envs\tf\lib\async